In [1]:
import os 
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
import nest_asyncio



load_dotenv()
memory = MemorySaver()


In [2]:
class State(TypedDict):
    user_input:str
    messages: Annotated[list, add_messages]
    urls:list[str]
    extracted:list[str]
    

In [ ]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [5]:
tool = TavilySearch(max_results=2)
tools = [tool]

In [6]:
tools_res = tool.invoke("What's a 'node' in LangGraph?")

In [8]:
tools_res["results"]

[{'title': 'Nodes and Edges | langchain-ai/langgraph-101 | DeepWiki',
  'url': 'https://deepwiki.com/langchain-ai/langgraph-101/2.2-nodes-and-edges',
  'content': 'Nodes and Edges | langchain-ai/langgraph-101 | DeepWiki Nodes and Edges Nodes and Edges What are Nodes and Edges? In LangGraph, a graph is composed of nodes connected by edges to form a directed workflow. Nodes are the workhorses of LangGraph - they are Python functions that receive the current graph state as input, perform operations, and return updates to that state. Edges define the flow of execution between nodes in a LangGraph. graph_builder.add_edge("retrieve_documents", "generate_response") Conditional edges use a function to determine the next node based on the current state. Building a Graph with Nodes and Edges graph_builder.add_node("retrieve_documents", retrieve_documents) graph_builder.add_edge("retrieve_documents", "generate_response") When designing nodes and edges in LangGraph: Nodes and Edges What are Nodes 

In [9]:
titles = []
urls = []
content=[]

for result in tools_res["results"]:
    titles.append(result["title"])
    urls.append(result["url"])
    content.append(result["content"])

In [10]:
urls

['https://deepwiki.com/langchain-ai/langgraph-101/2.2-nodes-and-edges',
 'https://www.geeksforgeeks.org/machine-learning/what-is-langgraph/']

In [19]:
def web_search(state:State):
    user_input=state["user_input"]
    tools_res = tool.invoke(user_input)
    
    urls = []

    for result in tools_res["results"]:
        urls.append(result["url"])
        
    return{
        "urls":urls,
    }

In [ ]:
sg=os.getenv("SG")
from scrapegraph_py import Client
client = Client(api_key=sg)

In [22]:
def scrap_data(state:State):
    urls = state["url"]
    
    
    res = []
    for i in urls:
        response = client.smartscraper(
        website_url=i,
        user_prompt="Extract everything about this provided link"
        )
        
        res.append(response)
        
    return {
        "extracted":res
    }

In [24]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["extracted"])]}

In [25]:
graph_builder = StateGraph(State)


graph_builder.add_node("search",web_search)
graph_builder.add_node("extract",scrap_data)
graph_builder.add_node("bot",chatbot)

graph_builder.set_entry_point("search")
graph_builder.add_edge("search","extract")
graph_builder.add_edge("extract","bot")
graph_builder.set_finish_point("bot")


graph = graph_builder.compile(checkpointer=memory)